---New---

In [2]:
# Cell 1 — Mount Drive and configure both datasets
from google.colab import drive
from pathlib import Path
import csv, json, os, shutil, subprocess, zipfile

os.chdir("/content")
drive.mount("/content/drive", force_remount=True)

BASE = Path("/content/drive/MyDrive/FLARE2026")
RAW_ROOT = BASE
WORK_ROOT = BASE / "AutoMSC2026"
SUBMISSION_ROOT = BASE / "automsc_submission"
DRIVE_CODEBASE = SUBMISSION_ROOT / "codebase"
LOCAL_CODEBASE = Path("/content/automsc-codebase")

DATASETS = ("Dataset008_EGCT", "Dataset009_PHLF")


# Copy code to Colab's faster local disk.
if LOCAL_CODEBASE.exists():
    shutil.rmtree(LOCAL_CODEBASE)

shutil.copytree(DRIVE_CODEBASE, LOCAL_CODEBASE)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(SUBMISSION_ROOT / "predictions").mkdir(parents=True, exist_ok=True)

os.environ["UV_PROJECT_ENVIRONMENT"] = "/content/automsc-uv-cu128"
os.environ["MPLBACKEND"] = "Agg"
os.chdir(LOCAL_CODEBASE)

for dataset in DATASETS:
    folder = RAW_ROOT / dataset
    for required in (
        "dataset.json",
        "cls_data.csv",
        "imagesTr",
        "labelsTr",
        "imagesTs",
    ):
        assert (folder / required).exists(), f"Missing: {folder / required}"


def ensure_best_segmentation_manifest(dataset):
    model_name = (
        "nnUNetTrainerQuickSeg__"
        "nnUNetResEncUNetMPlans__3d_fullres"
    )

    manifest_dir = WORK_ROOT / "manifests"
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / f"{dataset}.segmentation.json"

    if not manifest_path.is_file():
        checkpoint = (
            WORK_ROOT
            / "nnUNet_results"
            / dataset
            / model_name
            / "fold_all"
            / "checkpoint_best.pth"
        )
        assert checkpoint.is_file(), (
            f"Best segmentation checkpoint not found: {checkpoint}"
        )

        manifest = {
            "version": 1,
            "dataset_name": dataset,
            "configuration": "3d_fullres",
            "planner": "nnUNetPlannerResEncM",
            "plans_identifier": "nnUNetResEncUNetMPlans",
            "trainer": "nnUNetTrainerQuickSeg",
            "fold": "all",
            "model_name": model_name,
            "checkpoint_name": "checkpoint_best.pth",
            "checkpoint_path": str(checkpoint),
        }
        manifest_path.write_text(
            json.dumps(manifest, indent=2) + "\n"
        )

    selected = Path(
        json.loads(manifest_path.read_text())["checkpoint_path"]
    )
    assert (
        selected.name == "checkpoint_best.pth"
        and selected.is_file()
    ), f"Invalid segmentation checkpoint: {selected}"

    print(f"{dataset}: segmentation = {selected}")


def full_data_classification_is_complete(dataset):
    manifest_path = (
        WORK_ROOT
        / "manifests"
        / f"{dataset}.classification.json"
    )

    if not manifest_path.is_file():
        return False

    manifest = json.loads(manifest_path.read_text())
    tasks = manifest.get("tasks", [])

    return (
        manifest.get("version") == 5
        and bool(tasks)
        and all(
            task.get("training_scope") == "all_labelled_cases"
            and task.get("checkpoint_name") == "classifier_final.pth"
            and "models" not in task
            and Path(task.get("checkpoint_path", "")).is_file()
            for task in tasks
        )
    )


print("Datasets:", DATASETS)
print("Codebase:", DRIVE_CODEBASE)
print("Work root:", WORK_ROOT)
print("Prediction root:", SUBMISSION_ROOT / "predictions")

Mounted at /content/drive
Datasets: ('Dataset008_EGCT', 'Dataset009_PHLF')
Codebase: /content/drive/MyDrive/FLARE2026/automsc_submission/codebase
Work root: /content/drive/MyDrive/FLARE2026/AutoMSC2026
Prediction root: /content/drive/MyDrive/FLARE2026/automsc_submission/predictions


In [3]:
# Cell 2 — Install the locked environment
!uv sync --python 3.12
!nvidia-smi
!uv run python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA build:', torch.version.cuda); assert torch.cuda.is_available(), 'Select a GPU runtime'; print('GPU:', torch.cuda.get_device_name(0))"

Using CPython 3.12.14
Creating virtual environment at: /content/automsc-uv-cu128
Resolved 118 packages in 17ms
Prepared 107 packages in 28.10s
Installed 107 packages in 136ms
 + acvl-utils==0.2.6
 + annotated-types==0.8.0
 + anyio==4.15.0
 + argparse==1.4.0
 + batchgenerators==0.25.3
 + batchgeneratorsv2==0.3.5
 + blosc2==4.12.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + connected-components-3d==4.1.0
 + contourpy==1.3.3
 + cycler==0.12.1
 + dynamic-network-architectures==0.4.4
 + einops==0.8.2
 + filelock==3.32.5
 + flare-automsc-pipeline==1.2.0 (from file:///content/automsc-codebase)
 + fonttools==4.64.0
 + fsspec==2026.7.0
 + future==1.0.0
 + graphviz==0.21
 + h11==0.16.0
 + h2==4.4.1
 + hf-xet==1.6.0
 + hpack==4.2.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.30.0
 + hyperframe==6.1.0
 + idna==3.19
 + imagecodecs==2026.1.14
 + imageio==2.37.4
 + jinja2==3.1.6
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + lazy-loader==0.5
 + 

In [3]:
# classifier for Dataset009
DATASET = "Dataset009_PHLF"

ensure_best_segmentation_manifest(DATASET)

if full_data_classification_is_complete(DATASET):
    print(f"{DATASET}: v1.2 full-data classifier already exists; skipping")
else:
    subprocess.run([
        "uv", "run", "automsc", "train_cls", DATASET,
        "--raw-root", str(RAW_ROOT),
        "--work-root", str(WORK_ROOT),
        "--device", "cuda",
        "--epochs", "250",
        "--rebuild-cache",
    ], check=True)

Dataset009_PHLF: segmentation = /content/drive/MyDrive/FLARE2026/AutoMSC2026/nnUNet_results/Dataset009_PHLF/nnUNetTrainerQuickSeg__nnUNetResEncUNetMPlans__3d_fullres/fold_all/checkpoint_best.pth


In [4]:
# Infer and verify Dataset009
DATASET = "Dataset009_PHLF"
PREDICTION_ROOT = SUBMISSION_ROOT / "predictions"
subprocess.run([
        "uv", "run", "automsc", "infer", DATASET,
        "--raw-root", str(RAW_ROOT), "--work-root", str(WORK_ROOT),
        "--output-root", str(PREDICTION_ROOT), "--device", "cuda",
], check=True)
output = PREDICTION_ROOT / f"{DATASET}_prediction"
with (output / "predictions.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))
masks = list(output.glob("*.nii.gz"))
print(f"{DATASET}: {len(rows)} CSV rows, {len(masks)} masks, columns={list(rows[0]) if rows else []}")
print("Output:", output)

Dataset009_PHLF: 40 CSV rows, 40 masks, columns=['case_id', 'label']
Output: /content/drive/MyDrive/FLARE2026/automsc_submission/predictions/Dataset009_PHLF_prediction


In [4]:
# classifier for Dataset008
DATASET = "Dataset008_EGCT"

ensure_best_segmentation_manifest(DATASET)

if full_data_classification_is_complete(DATASET):
    print(f"{DATASET}: v1.2 full-data classifier already exists; skipping")
else:
    subprocess.run([
        "uv", "run", "automsc", "train_cls", DATASET,
        "--raw-root", str(RAW_ROOT),
        "--work-root", str(WORK_ROOT),
        "--device", "cuda",
        "--epochs", "250",
        "--rebuild-cache",
    ], check=True)

Dataset008_EGCT: segmentation = /content/drive/MyDrive/FLARE2026/AutoMSC2026/nnUNet_results/Dataset008_EGCT/nnUNetTrainerQuickSeg__nnUNetResEncUNetMPlans__3d_fullres/fold_all/checkpoint_best.pth


In [5]:
# Infer and verify Dataset008
DATASET = "Dataset008_EGCT"
PREDICTION_ROOT = SUBMISSION_ROOT / "predictions"
subprocess.run([
        "uv", "run", "automsc", "infer", DATASET,
        "--raw-root", str(RAW_ROOT), "--work-root", str(WORK_ROOT),
        "--output-root", str(PREDICTION_ROOT), "--device", "cuda",
], check=True)
output = PREDICTION_ROOT / f"{DATASET}_prediction"
with (output / "predictions.csv").open(newline="") as handle:
    rows = list(csv.DictReader(handle))
masks = list(output.glob("*.nii.gz"))
print(f"{DATASET}: {len(rows)} CSV rows, {len(masks)} masks, columns={list(rows[0]) if rows else []}")
print("Output:", output)

Dataset008_EGCT: 129 CSV rows, 129 masks, columns=['case_id', 'label_0', 'label_1', 'label_2']
Output: /content/drive/MyDrive/FLARE2026/automsc_submission/predictions/Dataset008_EGCT_prediction
